# google/flan-t5-base — model exploratory

**Three questions**
1. What is this model made of, and where would an embedding map attach?
2. How does it cut a Vietnamese/English dish name into tokens?
3. With no fine-tuning, can it copy a sentence — and where does it hesitate?

Pairing rule from the mission: this model is compared **base with base** (`vit5-base`), never across sizes.

In [1]:
import csv, json, os, sys, warnings
warnings.filterwarnings("ignore")
os.environ.setdefault("TQDM_DISABLE", "1")
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name != "research-prototype" else Path.cwd()
sys.path.insert(0, str(ROOT))
import torch
from src import config as C, models as M, data as D

def read_csv(name):
    p = ROOT / "results" / name
    if not p.exists():
        print(f"missing {p.name} - run: python -m src.run_all (stage not executed yet)")
        return []
    return list(csv.DictReader(p.open(encoding="utf-8")))

print("python", sys.version.split()[0], "| torch", torch.__version__,
      "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")


python 3.12.10 | torch 2.7.1+cu128 | cuda True | NVIDIA GeForce RTX 3050 Laptop GPU


## 1. Load — and print the device

`res-33`: a run that 'used the GPU' is a claim until the device string is printed.

In [2]:
tok, model, info = M.load("flan-t5-base")
info

loaded flan-t5-base     247.6M params  d_model=768 enc/dec=12/12  device=cuda dtype=bfloat16 vram=485 MB  3.75s


{'model_key': 'flan-t5-base',
 'hf_id': 'google/flan-t5-base',
 'd_model': 768,
 'layers_enc': 12,
 'layers_dec': 12,
 'heads': 12,
 'vocab': 32128,
 'params': 247577856,
 'dtype': 'bfloat16',
 'device': 'cuda',
 'load_s': 3.75,
 'vram_mb': 485.1,
 'pad_id': 0,
 'eos_id': 1}

## 2. Architecture — the surface a mapping would touch

In [3]:
cfg = model.config
rows = [("d_model", cfg.d_model), ("encoder layers", cfg.num_layers),
        ("decoder layers", cfg.num_decoder_layers), ("heads", cfg.num_heads),
        ("d_ff", cfg.d_ff), ("vocab", cfg.vocab_size),
        ("relative buckets", cfg.relative_attention_num_buckets), ("max positions", cfg.n_positions)]
enc = sum(p.numel() for p in model.encoder.parameters())
dec = sum(p.numel() for p in model.decoder.parameters())
emb = model.shared.weight.numel()
print(f"{'property':<20}{'value':>12}")
for k, v in rows: print(f"{k:<20}{v:>12}")
print(f"{'params enc':<20}{enc/1e6:>10.1f}M")
print(f"{'params dec':<20}{dec/1e6:>10.1f}M")
print(f"{'params emb (shared)':<20}{emb/1e6:>10.1f}M")

property                   value
d_model                      768
encoder layers                12
decoder layers                12
heads                         12
d_ff                        2048
vocab                      32128
relative buckets              32
max positions                512
params enc               109.6M
params dec               137.9M
params emb (shared)       24.7M


**Read this as:** every encoder layer outputs a `[seq, 768]` matrix. The paper's map
sits between two sequences of such matrices, so the number of *candidate attach points* is
`layers + 1` per side, and the two models must agree on `d_model` (768 → 768).

## 3. Tokenizer — a dish name is a span, not a token

In [4]:
samples = D.build(force=False)
lang = C.MODELS["flan-t5-base"]["lang"]
texts = [r[lang] for r in samples]
dishes = [r["dish_vi"] if lang == "vi" else r["dish_en"] for r in samples]

tpw = [len(tok(t, add_special_tokens=False)["input_ids"]) / max(1, len(t.split())) for t in texts]
dtok = [len(tok(d, add_special_tokens=False)["input_ids"]) for d in dishes]
print(f"tokens per word      : mean {sum(tpw)/len(tpw):.2f}  max {max(tpw):.2f}")
print(f"dish-name subwords   : mean {sum(dtok)/len(dtok):.2f}  max {max(dtok)}")

ids = tok('Today I ate pho bo with banh mi at my usual place.', add_special_tokens=False)["input_ids"]
print("\nsentence:", 'Today I ate pho bo with banh mi at my usual place.')
print("pieces  :", " | ".join(p.replace("\u2581", "\u2423") for p in tok.convert_ids_to_tokens(ids)))

OK  100 pairs | csv -> data\culinary_100_pairs.csv
    vi chars 5540 | en chars 6997
tokens per word      : mean 1.61  max 2.67
dish-name subwords   : mean 5.15  max 13

sentence: Today I ate pho bo with banh mi at my usual place.
pieces  : ␣Today | ␣I | ␣ | ate | ␣ | pho | ␣bo | ␣with | ␣ban | h | ␣mi | ␣at | ␣my | ␣usual | ␣place | .


Vietnamese words are split more finely than English ones (diacritics + a EN vocabulary),
so a dish name can be 1–6 subword pieces. **Consequence for measurement:** "perplexity of the dish name"
must be computed over a token *span*, not a single token — which is what `src/score.py` does.

## 4. One forward pass — what the layers actually hold

In [5]:
enc_in = tok(['Today I ate pho bo with banh mi at my usual place.'], return_tensors="pt").to(M.device())
with torch.no_grad():
    enc_out = model.encoder(**enc_in, output_hidden_states=True)
hs = enc_out.hidden_states
print("hidden states returned:", len(hs), "| shape:", tuple(hs[0].shape))
for i, h in enumerate(hs):
    print(f"layer {i:2d}  pooled-mean norm {h[0].mean(0).norm():7.2f}  last-token norm {h[0][-1].norm():7.2f}")

hidden states returned: 13 | shape: (1, 17, 768)
layer  0  pooled-mean norm   58.50  last-token norm  274.00
layer  1  pooled-mean norm  131.00  last-token norm 1840.00
layer  2  pooled-mean norm 8832.00  last-token norm 150528.00
layer  3  pooled-mean norm 15744.00  last-token norm 268288.00
layer  4  pooled-mean norm 15872.00  last-token norm 270336.00
layer  5  pooled-mean norm 15744.00  last-token norm 270336.00
layer  6  pooled-mean norm 15744.00  last-token norm 270336.00
layer  7  pooled-mean norm 15744.00  last-token norm 270336.00
layer  8  pooled-mean norm 15744.00  last-token norm 270336.00
layer  9  pooled-mean norm 19072.00  last-token norm 329728.00
layer 10  pooled-mean norm 19072.00  last-token norm 329728.00
layer 11  pooled-mean norm 19200.00  last-token norm 335872.00
layer 12  pooled-mean norm    1.55  last-token norm    0.95


The norm grows with depth, and the last token of the encoder is the summary a decoder would
attend to. If you later probe "which layer carries the dish meaning", these 13 matrices are the grid.

## 5. Zero-shot behaviour (no fine-tuning)

In [6]:
def gen(prompt, n=64):
    ids = tok(prompt, return_tensors="pt").to(M.device())
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=n, num_beams=1)
    return tok.decode(out[0], skip_special_tokens=True)

for p in ['Say: Pho is a famous Vietnamese dish.', 'translate Vietnamese to English: Phở bò là món ăn nổi tiếng của Việt Nam.', '1 + 3 = ?']:
    print("PROMPT:", p)
    print("OUTPUT:", gen(p))
    print()

PROMPT: Say: Pho is a famous Vietnamese dish.


OUTPUT: ng th ng th ng th ng th ng th ng th ng th ng th ng th ng th ng 

PROMPT: translate Vietnamese to English: Phở bò là món ăn nổi tiếng của Việt Nam.


OUTPUT: It is a very difficult task in Viet Nam.

PROMPT: 1 + 3 = ?
OUTPUT: 1



## 6. Copy test — 2,000 sentences

In [7]:
rows = read_csv("copy_flan-t5-base.csv")
if rows:
    bad = [r for r in rows if r["exact"] == "0"][:5]
    print(f"exact copies: {sum(r['exact'] == '1' for r in rows)} / {len(rows)}")
    for r in bad:
        print("SRC:", r["src"][:110])
        print("HYP:", r["hyp"][:110])
        print()
    s = [r for r in read_csv("copy_summary.csv") if r["model"] == "flan-t5-base"]
    if s:
        print({k: s[0][k] for k in ("n", "exact_match", "exact_match_norm", "bleu", "chrf",
                                   "tok_recall", "mean_hyp_words", "mean_ref_words")})

exact copies: 58 / 2000
SRC: The passion that the person has for her own growth is the most important thing .
HYP: She is a person who is passionate about her own growth.

SRC: Aware of the devastation and the challenges , I was keen among many other women to rebuild the Libyan civil so
HYP: I was a woman .

SRC: As you can see , the bulbs face outside , because that 's where the lions come from .
HYP: The lions are a lion .

SRC: I am on this stage because I am a pretty , white woman , and in my industry we call that a sexy girl .
HYP: I am a pretty white woman .

SRC: This is when we started asking passing tourists to take the picture .
HYP: We were a group of tourists .

{'n': '2000', 'exact_match': '0.0265', 'exact_match_norm': '0.029', 'bleu': '8.4', 'chrf': '23.44', 'tok_recall': '0.3358', 'mean_hyp_words': '8.43', 'mean_ref_words': '21.09'}


## 7. Perplexity — is the dish name harder than the rest of the sentence?

In [8]:
s = [r for r in read_csv("ppl_summary.csv") if r["model"] == "flan-t5-base"]
if s:
    r = s[0]
    print({k: r[k] for k in ("ppl_all", "ppl_dish", "ppl_other", "rank_dish", "rank_other",
                            "delta_nll", "delta_ci_lo", "delta_ci_hi", "dish_harder_sentences", "sentences")})
toks = read_csv("ppl_tokens_flan-t5-base.csv")
if toks:
    print("\nhardest tokens (highest NLL):")
    for r in sorted(toks, key=lambda x: -float(x["nll"]))[:10]:
        print(f"  {float(r['nll']):6.2f}  rank {int(r['rank']):>6}  dish={r['in_dish']}  {r['span']!r}")

{'ppl_all': '1.17', 'ppl_dish': '1.07', 'ppl_other': '1.2', 'rank_dish': '1.0', 'rank_other': '1.3', 'delta_nll': '-0.107', 'delta_ci_lo': '-0.144', 'delta_ci_hi': '-0.068', 'dish_harder_sentences': '22', 'sentences': '100'}

hardest tokens (highest NLL):
    7.93  rank     82  dish=0  ' mixes'
    7.49  rank    256  dish=1  'Com'
    7.44  rank    207  dish=1  'Rau'
    6.22  rank     64  dish=0  ' road'
    6.18  rank     58  dish=1  'Thi'
    5.78  rank     25  dish=0  ' blended'
    5.78  rank     42  dish=0  ' bought'
    5.65  rank     23  dish=0  'Southern'
    5.54  rank     29  dish=0  ' stew'
    5.53  rank     27  dish=0  ' loved'


`delta_nll > 0` means dish tokens are *less* predictable than the sentence around them.
The CI is a paired bootstrap over the 100 sentences, so it answers "is the dish harder **in the same sentence**",
not "is this model worse than that model".

## 8. What this means for the paper

- The *copy gate* is answered in §6: if `exact_match` is low, fine-tuning is not optional — the model will not
  even reproduce its own input, so a mapped vector cannot be read out.
- The *attach surface* in §2/§4 is `[seq, 768]` at 13 depths; the pairing rule keeps `base` with `base`.
- The *dish span* in §3/§7 is the unit the paper's perplexity story has to be told in.